# SQL Agent with LangGraph — Walmart Sales
**Goal:** Introduction to LangGraph DAGs for SQL querying on Walmart Sales data

## Libraries

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain

# LangGraph
from langgraph.graph import END, StateGraph
from typing import TypedDict

import pandas as pd
import sqlalchemy as sql
import os
import re
import yaml
from pprint import pprint
from IPython.display import Markdown

## AI Setup

In [ ]:
os.environ["OPENAI_API_KEY"] = yaml.safe_load(open('../credentials.yml'))['openai']

llm = ChatOpenAI(model="gpt-4o-mini")

## 1.0 SQL Agent — Walmart Sales Database

In [ ]:
PATH_DB = "sqlite:///../data/walmart_sales.db"

sql_engine = sql.create_engine(PATH_DB)
conn = sql_engine.connect()

db = SQLDatabase.from_uri(PATH_DB)

sql_generator = create_sql_query_chain(
    llm=llm,
    db=db,
    k=int(1e7),
)

print("Tables:", db.get_usable_table_names())

## 2.0 SQL Parsing Utility

In [ ]:
def extract_sql_code(text: str):
    """
    Extracts the SQL query from a block of text.
    """
    patterns = [
        r"SQLQuery:\s*```sql\s*(?P<sql>[\s\S]+?)```",
        r"```sql\s*(?P<sql>[\s\S]+?)```",
        r"```(?:[\s\S]*?)\s*(?P<sql>SELECT[\s\S]+?)```",
        r"SQLQuery:\s*(?P<sql>[\s\S]+?)(?=\n\s*\n|$)",
        r"(?P<sql>SELECT[\s\S]+?;)(?=\s|$)",
    ]
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE)
        if m:
            sql_code = m.group("sql").strip()
            if sql_code.startswith(('"', "'")) and sql_code.endswith(('"', "'")):
                sql_code = sql_code[1:-1].strip()
            return sql_code
    return None

In [ ]:
# Test the generator
response = sql_generator.invoke({"question": "Which 10 items have the highest total demand value?"})
sql_q = extract_sql_code(response)
pprint(sql_q)
pd.read_sql(sql_q, conn)

## 3.0 Build a LangGraph DAG

In [ ]:
class GraphState(TypedDict):
    """Represents the state of our graph."""
    question: str
    sql_query: str


def generate_sql(state):
    print("---GENERATE SQL---")
    question = state.get("question")
    sql_query = sql_generator.invoke({"question": question})
    sql_query = extract_sql_code(sql_query)
    return {"sql_query": sql_query}


def state_printer(state):
    """Print the state."""
    print("---STATE PRINTER---")
    print(f"question: {state.get('question')}")
    pprint(f"SQL Query:\n{state.get('sql_query')}")


# Build the workflow DAG
workflow = StateGraph(GraphState)

workflow.add_node("generate_sql", generate_sql)
workflow.add_node("state_printer", state_printer)

workflow.set_entry_point("generate_sql")
workflow.add_edge("generate_sql", "state_printer")
workflow.add_edge("state_printer", END)

app = workflow.compile()
app

## 4.0 Testing the Graph

In [ ]:
QUESTION = "Which 10 items have the highest total demand value?"

response = app.invoke({"question": QUESTION})
print("SQL:", response['sql_query'])
db.run(response['sql_query'])

In [ ]:
QUESTION = "What are the total sales value by year-month? Order by date."

response = app.invoke({"question": QUESTION})
sql_q = response.get("sql_query")
pprint(sql_q)
pd.read_sql(sql_q, conn)

In [ ]:
QUESTION = "What is the total demand value by item_id? Return the top 20 items ordered by total value descending."

response = app.invoke({"question": QUESTION})
sql_q = response.get("sql_query")
pprint(sql_q)
pd.read_sql(sql_q, conn)